## Data Validation 

In [1]:
import os
%pwd

'd:\\Data Science\\END to END Proj\\Introvert vs Extrovert\\Introvert-Vs-Extrovert\\research'

In [2]:
os.chdir("../")

In [3]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class DataValidationConfig:
    root_dir: Path
    STATUS_FILE: str
    train_data_path: Path
    original_data_path: Path
    clean_train_path: Path          # ➜ new
    clean_org_path: Path            # ➜ new
    all_schema: dict



In [4]:
from src.IntrovertVsExtrovert.utils.common import read_yaml, create_directories
from src.IntrovertVsExtrovert.constant import *

class ConfigurationManager:
    def __init__(
        self,
        config_filepath=CONFIG_FILE_PATH,
        params_filepath=PARAMS_FILE_PATH,
        schema_filepath=SCHEMA_FILE_PATH
    ):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)

        create_directories([self.config.artifacts_root])

    def get_data_validation_config(self) -> DataValidationConfig:
        config = self.config.data_validation
        schema = self.schema.COLUMNS

        create_directories([config.root_dir])

        return DataValidationConfig(
            root_dir=config.root_dir,
            STATUS_FILE=config.STATUS_FILE,
            train_data_path=config.train_data_path,
            original_data_path=config.original_data_path,
            clean_train_path=config.clean_train_path,     
            clean_org_path=config.clean_org_path,         
            all_schema=schema
        )



In [5]:
import pandas as pd

class DataValidation:
    def __init__(self, config: DataValidationConfig):
        self.config = config

    def _check_schema(self, df: pd.DataFrame, dataset_name: str) -> bool:
        actual_columns = set(df.columns)
        expected_columns = set(self.config.all_schema.keys())

        if not expected_columns.issubset(actual_columns):
            missing = expected_columns - actual_columns
            print(f"[{dataset_name}] ❌ Missing columns: {missing}")
            return False

        return True

    def _check_missing_and_duplicates(self, df: pd.DataFrame, dataset_name: str) -> pd.DataFrame:
        print(f"\n[{dataset_name}] 🧹 Checking for missing values...")
        print(df.isnull().sum())

        print(f"\n[{dataset_name}] 🧹 Checking for duplicates...")
        dup_count = df.duplicated().sum()
        print(f"Found {dup_count} duplicated rows.")

        if dup_count > 0:
            df = df.drop_duplicates()
            print(f"[{dataset_name}] ✅ Duplicates dropped.")

        return df

    def validate_and_clean(self):
        try:
            train_df = pd.read_csv(self.config.train_data_path)
            org_df = pd.read_csv(self.config.original_data_path)

            # Drop id column if present
            if 'id' in train_df.columns:
                train_df.drop(columns=['id'], inplace=True)

            # Schema check
            status_train = self._check_schema(train_df, "Train Data")
            status_org = self._check_schema(org_df, "Original Data")

            validation_status = status_train and status_org

            # Clean datasets
            if validation_status:
                train_df = self._check_missing_and_duplicates(train_df, "Train Data")
                org_df = self._check_missing_and_duplicates(org_df, "Original Data")
                train_df.to_csv(self.config.clean_train_path, index=False)
                org_df.to_csv(self.config.clean_org_path, index=False)

            # Save validation result
            with open(self.config.STATUS_FILE, 'w') as f:
                f.write(f"Validation status: {validation_status}")

            return train_df, org_df

        except Exception as e:
            raise e


In [6]:
try:
    config = ConfigurationManager()
    data_validation_config = config.get_data_validation_config()

    validator = DataValidation(config=data_validation_config)
    cleaned_train_df, cleaned_org_df = validator.validate_and_clean()

    print("✅ Data Validation and Cleaning Successful.")
except Exception as e:
    print(f"❌ Exception during validation: {e}")


[2025-07-12 11:33:07,844: INFO: common: yaml file: config\config.yaml loaded successfully]


[2025-07-12 11:33:07,874: INFO: common: yaml file: params.yaml loaded successfully]
[2025-07-12 11:33:07,899: INFO: common: yaml file: schema.yaml loaded successfully]
[2025-07-12 11:33:07,907: INFO: common: created directory at: artifacts]
[2025-07-12 11:33:07,909: INFO: common: created directory at: artifacts/data_validation]

[Train Data] 🧹 Checking for missing values...
Time_spent_Alone             1190
Stage_fear                   1893
Social_event_attendance      1180
Going_outside                1466
Drained_after_socializing    1149
Friends_circle_size          1054
Post_frequency               1264
Personality                     0
dtype: int64

[Train Data] 🧹 Checking for duplicates...
Found 0 duplicated rows.

[Original Data] 🧹 Checking for missing values...
Time_spent_Alone             63
Stage_fear                   73
Social_event_attendance      62
Going_outside                66
Drained_after_socializing    52
Friends_circle_size          77
Post_frequency              